# Fine-tune do embedder com dados do seu WhatsApp SaaS

Este notebook treina um modelo customizado pra **sua loja** usando os dados
que o sistema acumulou. Resultado: busca semântica que entende a linguagem
específica dos seus clientes.

## Pré-requisitos

1. Pelo menos **500 interações com `outcome` positivo** (`purchased`, `added_to_cart`, `asked_more`).
   Cheque em: `GET /dev/training-data/stats`
2. Modelo base que você está usando em produção (`bge-m3`).
3. Runtime do Colab com **GPU** (Runtime → Change runtime type → T4 GPU). 15-30 min de treino.

## Como rodar

1. Faça upload do `triplets.jsonl` (gerado pelo endpoint `/dev/training-data/export`).
2. Rode todas as células sequencialmente.
3. No fim, baixa um modelo `.zip`. Sobe no servidor e troca em `OLLAMA_EMBEDDING_MODEL`.

## 1. Setup

In [ ]:
!pip install -q sentence-transformers==3.3.1 datasets==3.2.0 huggingface-hub==0.27.0

In [ ]:
from google.colab import files
uploaded = files.upload()  # selecione triplets.jsonl

## 2. Carrega dataset

In [ ]:
import json
from datasets import Dataset

FILENAME = next(iter(uploaded.keys()))
rows = []
with open(FILENAME) as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        rows.append(json.loads(line))

print(f"Total triplets: {len(rows)}")
print("Sample:", rows[0] if rows else None)

if len(rows) < 200:
    print("\n⚠️  Menos de 200 triplets é pouco. Treine mas espere ganho marginal.")
    print("   Recomendado: esperar acumular mais dados.")

# Sentence-transformers v3 espera colunas 'anchor', 'positive', 'negative'
ds = Dataset.from_list(rows)
print(ds)

## 3. Carrega modelo base

**bge-m3** roda em ~2GB de VRAM. Cabe em GPU T4 grátis.

In [ ]:
from sentence_transformers import SentenceTransformer

BASE_MODEL = 'BAAI/bge-m3'
model = SentenceTransformer(BASE_MODEL, device='cuda')
print(f"Modelo: {BASE_MODEL}")
print(f"Dimensões: {model.get_sentence_embedding_dimension()}")

## 4. Treino

**MultipleNegativesRankingLoss** é o padrão pra triplets — ensina o modelo a aproximar
anchor↔positive e afastar anchor↔negative usando os outros itens do batch também
como negativos implícitos.

In [ ]:
from sentence_transformers import SentenceTransformerTrainer, SentenceTransformerTrainingArguments
from sentence_transformers.losses import MultipleNegativesRankingLoss

split = ds.train_test_split(test_size=0.1, seed=42)

args = SentenceTransformerTrainingArguments(
    output_dir='./embedder-finetuned',
    num_train_epochs=2,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    learning_rate=2e-5,
    warmup_ratio=0.1,
    fp16=True,
    eval_strategy='steps',
    eval_steps=50,
    save_strategy='steps',
    save_steps=50,
    save_total_limit=2,
    logging_steps=20,
    load_best_model_at_end=True,
    report_to=[],
)

trainer = SentenceTransformerTrainer(
    model=model,
    args=args,
    train_dataset=split['train'],
    eval_dataset=split['test'],
    loss=MultipleNegativesRankingLoss(model),
)
trainer.train()

## 5. Salva e baixa

Pra usar no Ollama, precisa converter pra GGUF. Caminho mais simples
no MVP: subir o modelo no HuggingFace Hub e usar via API (OpenAI-compatible).

Ou exportar e usar com sentence-transformers no próprio NestJS via Python sidecar.

**Recomendação prática**: começar usando esse modelo via API HF Inference Endpoints
ou hospedar com vLLM/text-embeddings-inference. Custo ~$0.10/hora numa GPU pequena.

In [ ]:
model.save('./embedder-finetuned/final')

import shutil
shutil.make_archive('embedder-finetuned', 'zip', './embedder-finetuned/final')
files.download('embedder-finetuned.zip')

## 6. Avaliação rápida (opcional, mas recomendado)

Antes de trocar em produção, valida que o modelo melhorou. Mede
o quanto a similaridade entre query positiva é > similaridade negativa.

In [ ]:
import torch
from sentence_transformers.util import cos_sim

test_rows = list(split['test'])
anchors = [r['anchor'] for r in test_rows]
positives = [r['positive'] for r in test_rows]
negatives = [r['negative'] for r in test_rows]

anchor_emb = model.encode(anchors, convert_to_tensor=True)
positive_emb = model.encode(positives, convert_to_tensor=True)
negative_emb = model.encode(negatives, convert_to_tensor=True)

pos_scores = torch.diagonal(cos_sim(anchor_emb, positive_emb))
neg_scores = torch.diagonal(cos_sim(anchor_emb, negative_emb))

wins = (pos_scores > neg_scores).float().mean().item()
print(f"% de queries onde positivo > negativo: {wins*100:.1f}%")
print("Idealmente >85%. Se <70%, dataset é muito ruidoso ou pequeno demais.")